In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- 1. モデル定義 ---
def model_with_tangent(z_all, t, params):
    x1, y1, x2, y2 = z_all[:4]
    w = z_all[4:]
    
    r_x1, r_y1, r_x2, r_y2 = params['r']
    lam = params['lambda']
    c = params['c']
    d = params['d']
    
    l11, l12 = lam[0]
    l21, l22 = lam[1]
    c1, c2 = c
    d1, d2 = d

    # 状態方程式
    dx1dt = r_x1*x1 - l11*x1*y1 - l12*x1*y2
    dx2dt = r_x2*x2 - l21*x2*y1 - l22*x2*y2
    dy1dt = -r_y1*y1 + c1*l11*x1*y1 + d1*l21*x2*y1
    dy2dt = -r_y2*y2 + c2*l12*x1*y2 + d2*l22*x2*y2
    
    # ヤコビ行列 J
    J = np.zeros((4, 4))
    J[0, 0] = r_x1 - l11*y1 - l12*y2
    J[0, 1] = -l11*x1
    J[0, 2] = 0
    J[0, 3] = -l12*x1
    
    J[1, 0] = c1*l11*y1
    J[1, 1] = -r_y1 + c1*l11*x1 + d1*l21*x2
    J[1, 2] = d1*l21*y1
    J[1, 3] = 0

    J[2, 0] = 0
    J[2, 1] = -l21*x2
    J[2, 2] = r_x2 - l21*y1 - l22*y2
    J[2, 3] = -l22*x2
    
    J[3, 0] = c2*l12*y2
    J[3, 1] = 0
    J[3, 2] = d2*l22*y2
    J[3, 3] = -r_y2 + c2*l12*x1 + d2*l22*x2

    dwdt = J @ w
    return np.array([dx1dt, dy1dt, dx2dt, dy2dt, dwdt[0], dwdt[1], dwdt[2], dwdt[3]])
# --- ★追加箇所: ルンゲ・クッタ法(RK4) ---
def rk4_step(func, z, t, dt, params):
    k1 = func(z, t, params)
    k2 = func(z + 0.5 * dt * k1, t + 0.5 * dt, params)
    k3 = func(z + 0.5 * dt * k2, t + 0.5 * dt, params)
    k4 = func(z + dt * k3, t + dt, params)
    return z + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
# --- 2. パラメータ設定 (0.0055が出る設定) ---
params = {
    'r': [1.5, 1.0, 1.0, 1.0],
    'lambda': [[1.0, 0.6], [0.6, 1.0]],
    'c': [1.0, 0.6],
    'd': [0.6, 0.8]
}

# --- 3. 計算実行 (履歴を保存するように修正) ---
t_max = 4000 # 少し短くても十分見えます
dt = 0.05
steps = int(t_max / dt)

z = np.array([2.0, 2.0, 1.5, 1.5]) 
w = np.array([1.0, 0.0, 0.0, 0.0])
w = w / np.linalg.norm(w)

cum_log_norm = 0.0
lle_history = []
z_history = [] # ★ここに軌道を保存します

transient_steps = 1000

print(f"Benettin法で計算中... (Total steps: {steps})")
z_curr = np.concatenate([z, w])
for i in range(steps):
    t = i * dt
    
    # --- ★変更箇所: odeintをやめてRK4関数を呼ぶ ---
    # 元: sol = odeint(model_with_tangent, z_all_init, t_span, args=(params,))
    z_next_all = rk4_step(model_with_tangent, z_curr, t, dt, params)
    
    z_next = z_next_all[:4]
    w_next = z_next_all[4:]
    
    # 軌道の保存 (あとでプロットするため)
    z_history.append(z_next)

    d_norm = np.linalg.norm(w_next)
    
    if i > transient_steps:
        cum_log_norm += np.log(d_norm)
        current_time = (i - transient_steps) * dt
        lle = cum_log_norm / current_time
        lle_history.append(lle)
    
    z = z_next
    w = w_next / d_norm

# リストをnumpy配列に変換 (これで2次元配列になります)
z_history = np.array(z_history)

print("-" * 30)
print(f"最終LLE: {lle_history[-1]:.6f}")
print("-" * 30)

# --- 4. プロット (修正版) ---
# 後半の安定した部分だけを取り出す
plot_start = 5000
if len(z_history) > plot_start:
    z_plot = z_history[plot_start:]
else:
    z_plot = z_history # データが短い場合は全部使う

fig = plt.figure(figsize=(12, 5))

# 左側：LLEの収束
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(lle_history)
ax1.set_title("LLE Convergence")
ax1.set_xlabel("Steps")
ax1.set_ylabel("LLE")
ax1.grid(True)

# 右側：3D軌道 (知恵の輪になっているか確認)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot(z_plot[:, 0], z_plot[:, 1], z_plot[:, 2], lw=0.5)
ax2.set_title("Phase Portrait (3D)")
ax2.set_xlabel("x1")
ax2.set_ylabel("y1")
ax2.set_zlabel("x2")

plt.tight_layout()
plt.show()

In [ ]:
# --- 5. ポアンカレ断面の描画 (Torus vs Chaos 判定) ---
# y1 の平均値をまたぐ瞬間を捕捉して、その時の x1, x2 をプロットします

# 安定した後半のデータを使用
z_stable = z_history[5000:] 

# 切断する平面の定義 (y1 = 平均値)
y_plane = np.mean(z_stable[:, 1])

crossings_x1 = []
crossings_x2 = []

# 軌道が平面を通過する点を補間して求める
for i in range(len(z_stable) - 1):
    y_current = z_stable[i, 1]
    y_next = z_stable[i+1, 1]
    
    # y_current < Plane < y_next (下から上へ通過) の瞬間を検知
    if (y_current < y_plane and y_next > y_plane) or (y_current > y_plane and y_next < y_plane):
        # 線形補間で交差点を正確に計算
        frac = (y_plane - y_current) / (y_next - y_current)
        
        x1_cross = z_stable[i, 0] + frac * (z_stable[i+1, 0] - z_stable[i, 0])
        x2_cross = z_stable[i, 2] + frac * (z_stable[i+1, 2] - z_stable[i, 2])
        
        crossings_x1.append(x1_cross)
        crossings_x2.append(x2_cross)

# プロット
plt.figure(figsize=(6, 6))
plt.plot(crossings_x1, crossings_x2, '.', markersize=2)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title(f'Poincare Section at y1={y_plane:.2f}\n(Circle=Quasi-periodic, Cloud=Chaos)')
plt.grid(True)
plt.axis('equal')
plt.show()